### Getting Started With Basic Langchain application
#### In this we will see how to:- 
* Get setup with LangChain, LangSmith, LangServe
* Use the Most basic and common components of LangChain like: Prompt templates, Models and the output parsers
* Build a simple application with LangChain
* Trace your application with LangSmith
* Serve your application with LangServe

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")

In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.15', 'langchain-google-genai': '4.3.5'}} profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'} google_api_key=SecretStr('**********') model='gemini-3.6-flash' temperature=None client=<google.genai.client.Client object at 0x0000018444B09600> default_metadata=() model_kwargs={}


In [13]:
result=llm.invoke("What is Generative AI in 20 words")
print(result.content)

[{'type': 'text', 'text': 'Generative AI is artificial intelligence that creates new content, such as text, images, and code, by learning patterns from data.', 'extras': {'signature': 'EqMfCqAfARFNMg/MaFhP+MdMc4YMTB6j3vex1OyJ69BRCU4ZqZaZSPZ2KP/TmMOO6UNaWn9IgjFuVW8XVDXT8Yv3QzdDKNhgOpCZUwn9gIfXtKu4iVB7Rogzbpm9R2yDhTWO+B68L8AhKVOBu8eFfK4hsohVK5AlNKmQScy0d6vpKdSdHibtlB07nhmOw8JoD9gqvb4M56qRQ3R9X/qyJ6Dh24iuxky5N1SZ85chqqZaVnpdOau4xPS9hMOPWbowlGlb2lqeLI3LWdNVQmzOlUq4KflWlYscYVQW24qhCeF3nhPJjfUcgk/DSuX1ZnIQBTXGV5ymlB7dFIFRl/3gl0gdx6/P7ehnFeZRlCtsAW81dL1rApyuVSAJFa+ib9UlkPWBBmlN2TRRqDVdOTY5A+5e/TBmFWVXE5Bxwr1lcJnovdtXCm0zw0mvnSh+yxQ90MynN5/HbeQRRz5vlrbjKQoNW/nJMeVXlnMJrZRVP2ERlYkGMVXfMFyuS0xzODCXYlySj5rUzsHIdwreq8uRIZ6M1FZfStu90DE1/azbjOON/crYFDOUr28CHN9Zh302FYDZdJV87h4oRr4/iK3AObc4mrur9KI4hofyKS3bPzOVHILanP6q6CG6/KUCdc13RrNJkq48JVfG6ZFs15ZsOuMwiNDL4AYoAiUAAuyZZ54ZXzojjs5Lmq9M1diITBtchcamAntCcTNax/Mimh/nZLcsSlJVTMkCvKpcPeDCFT+xxPqB+nF8vMNkD05Q9pZ1aalTeMpg9AleDBNgom/qw9RmqSoUXSoDTYElNT9DeLJ9rBy

In [15]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","yor are an expert AI engineer. Provide me answers based on the questions."),
        ("user","{input}")
    ]
)
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='yor are an expert AI engineer. Provide me answers based on the questions.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

### Chain

In [16]:
chain=prompt|llm
response=chain.invoke({"input":"Tell me about LangChain and LangGraph in short"})
print(response)

content=[{'type': 'text', 'text': 'Here is a concise, engineer-level summary of **LangChain** and **LangGraph**:\n\n---\n\n### 1. LangChain\n* **What it is:** A popular open-source framework for building applications powered by Large Language Models (LLMs).\n* **Core Concept:** **Linear chains (DAGs).** It connects components—like prompts, LLMs, vector databases, and memory—into step-by-step sequential pipelines.\n* **Best Used For:** Standard LLM tasks, Retrieval-Augmented Generation (RAG), basic data transformations, and quick prototyping.\n* **Limitation:** Struggles with complex, non-linear workflows that require loops, cycles, or state persistence (e.g., "try X, if it fails, try Y").\n\n---\n\n### 2. LangGraph\n* **What it is:** An extension of LangChain built specifically for creating **stateful, multi-agent AI applications**.\n* **Core Concept:** **Cyclic graphs.** It models workflows using **Nodes** (actions/agents), **Edges** (decision logic), and a shared **State**. It allows

In [17]:
type(response)

langchain_core.messages.ai.AIMessage

### Output Parser

In LangChain, an Output Parser is a component that takes the raw response generated by an 
LLM and converts it into a structured format that your Python program can easily use.

In [20]:
## Output Parser
from langchain_core.output_parsers import StrOutputParser
output_parser=StrOutputParser()
chain=prompt|llm|output_parser
response=chain.invoke({"input":"Tell me concise details about LLM."})
print(response)

Here is a high-level, technical overview of Large Language Models (LLMs) from an AI engineering perspective.

---

### 1. What is an LLM?
A **Large Language Model (LLM)** is a deep learning model trained on vast amounts of textual data to understand, generate, and process human language. At its core, an LLM is a probabilistic engine that predicts the **next most likely token** (word or sub-word) given a sequence of input tokens.

---

### 2. Core Architecture
Almost all modern LLMs are based on the **Transformer architecture** (introduced in 2017), characterized by:
*   **Self-Attention Mechanism:** Allows the model to dynamically weight the importance of different words in a sequence, regardless of their distance from each other.
*   **Decoder-Only Design:** Modern generative models (e.g., GPT-4, Llama 3, Mistral) primarily use decoder-only architectures optimized for autoregressive generation.

---

### 3. The Training Lifecycle
Building a production-grade LLM involves three main pha